# AI Mathematical Olympiad — Full Pipeline

**MCTS-based Math Solver with Symbolic Verification**

This notebook runs the complete pipeline:
1. Install dependencies & verify GPU
2. Data pipeline — stream OpenMathReasoning TIR → parquet
3. SFT training — QLoRA fine-tune NuminaMath-7B-TIR
4. MCTS-RL training — generate verified solutions via MCTS, fine-tune on them
5. Evaluation — compare base vs SFT vs MCTS-RL
6. Publication-quality plots
7. Download trained adapters

**Before running:** Runtime → Change runtime type → **T4 GPU**

---
## 1. Setup & GPU Verification

In [ ]:
!pip install -q transformers>=4.44.0 datasets peft bitsandbytes accelerate trl>=0.9.0 \
    matplotlib sympy pandas pyarrow tqdm

In [ ]:
import torch
import os

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

GPU_NAME  = torch.cuda.get_device_name(0)
VRAM_GB   = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")
print(f"PyTorch: {torch.__version__}")

# Directories
os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/models/sft", exist_ok=True)
os.makedirs("/content/models/mcts_rl", exist_ok=True)
os.makedirs("/content/experiments", exist_ok=True)

---
## 2. Data Pipeline — Stream OpenMathReasoning TIR

Streams from `nvidia/OpenMathReasoning` (TIR split), transforms to chat format,
and writes to a parquet file. Only the current batch is in memory.

In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from tqdm.auto import tqdm

DATA_LIMIT = 50_000  # Set to None for full 1.7M rows
BATCH_SIZE = 5_000
OUTPUT_PATH = "/content/data/processed/math_reasoning_tir.parquet"

SCHEMA = pa.schema([
    ("problem_id", pa.string()),
    ("problem", pa.string()),
    ("solution", pa.string()),
    ("expected_answer", pa.string()),
    ("difficulty", pa.string()),
    ("messages", pa.list_(pa.struct([
        ("role", pa.string()),
        ("content", pa.string()),
    ]))),
])

_row_counter = 0

def build_tir_messages(problem, solution):
    return [
        {"role": "system", "content": ""},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": solution},
    ]


def format_tir_chat(problem, solution):
    return (
        f"<|system|>\n<|end|>\n"
        f"<|user|>\n{problem}<|end|>\n"
        f"<|assistant|>\n{solution}<|end|>"
    )


def transform_row(example):
    global _row_counter
    _row_counter += 1
    problem_id = f"tir_{_row_counter:06d}"
    problem = example["problem"]
    solution = example["generated_solution"]
    return {
        "problem_id": problem_id,
        "problem": problem,
        "solution": solution,
        "expected_answer": example["expected_answer"],
        "difficulty": example.get("problem_source", "unknown"),
        "messages": build_tir_messages(problem, solution),
    }


print("Streaming nvidia/OpenMathReasoning (tir split)...")
ds = load_dataset("nvidia/OpenMathReasoning", split="tir", streaming=True)

writer = None
batch = []
rows_written = 0

for example in tqdm(ds, desc="Processing", total=DATA_LIMIT):
    if DATA_LIMIT and rows_written + len(batch) >= DATA_LIMIT:
        break
    batch.append(transform_row(example))

    if len(batch) >= BATCH_SIZE:
        table = pa.Table.from_pylist(batch, schema=SCHEMA)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
        writer.write_table(table)
        rows_written += len(batch)
        batch.clear()
        del table
        gc.collect()
        print(f"  Written {rows_written:,} rows")

if batch:
    table = pa.Table.from_pylist(batch, schema=SCHEMA)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
    writer.write_table(table)
    rows_written += len(batch)
    batch.clear()  # MEMORY FIX: Clear final batch
    del table

if writer:
    writer.close()

# MEMORY FIX: Clear dataset reference
del ds, writer
gc.collect()

final_count = pq.read_metadata(OUTPUT_PATH).num_rows
print(f"\nDone: {final_count:,} rows -> {OUTPUT_PATH}")

In [ ]:
# Quick peek at the data
import pandas as pd
df_peek = pd.read_parquet(OUTPUT_PATH, columns=["problem", "expected_answer", "difficulty"])
print(f"Total rows: {len(df_peek):,}")
print(f"\nDifficulty distribution:")
print(df_peek["difficulty"].value_counts().head(10))
print(f"\nSample problem:\n{df_peek.iloc[0]['problem'][:300]}...")